# Week 13 — Dataset B: 教育部高教統計

**用途**：在 VS Code / Jupyter 中一格一格 (cell-by-cell) 跑完整個 pipeline。

**對應 .py 檔**：`moe_higher_ed_pipeline.py`

**研究問題**：少子化對台灣高等教育的衝擊有多大？哪個 sector (公立 vs. 私立) 首當其衝？

**在 PowerShell 中跑這個檔案**：
```powershell
# 在 VS Code 中：開啟此 .ipynb，Shift+Enter 逐格執行
# 在 PowerShell 中跑整支 .py：
python .\moe_higher_ed_pipeline.py
```

## 1. 載入套件 + 設定 URL

In [4]:
from io import StringIO

# 教育部 stats.moe.gov.tw 的 SSL 憑證在 Python 內建的 OpenSSL 信任鏈下會
# 觸發 "Missing Subject Key Identifier" 錯誤。truststore 改用作業系統
# (Windows / macOS) 的憑證儲存區驗證,可以正常連線。必須在 import requests 之前呼叫。
import truststore
truststore.inject_into_ssl()

import pandas as pd
import plotly.express as px
import requests

# 教育部統計處每學年度的 CSV 都遵循同一個 URL 範本
MOE_URL = "https://stats.moe.gov.tw/files/detail/{year}/{year}_student.csv"
HEADERS = {"User-Agent": "NS5116-week13-teaching-example"}

print("套件載入完成")

套件載入完成


## 2. 抓一個學年度試試看

**注意**：教育部 CSV 用 UTF-8 with BOM 編碼，要用 `utf-8-sig` 解碼。

In [5]:
year = 113   # 113 學年度
url = MOE_URL.format(year=year)
r = requests.get(url, timeout=30, headers=HEADERS)
r.raise_for_status()

df_113 = pd.read_csv(StringIO(r.content.decode("utf-8-sig")))
print(f"113 學年度: shape = {df_113.shape}")
print("\n前幾欄：", list(df_113.columns)[:8])
df_113.head()

113 學年度: shape = (737, 28)

前幾欄： ['學年度', '學校代碼', '學校名稱', '日間∕進修別', '等級別', '總計', '男生計', '女生計']


,學年度,學校代碼,學校名稱,日間∕進修別,等級別,總計,男生計,女生計,華語先修生男,華語先修生女,...,五年級男,五年級女,六年級男,六年級女,七年級男,七年級女,延修生男,延修生女,縣市名稱,體系別
0,113,1,國立政治大學,D 日,D 博士,897,482,415,0,0,...,61,62,47,59,72,44,0,0,30 臺北市,1 一般
1,113,1,國立政治大學,D 日,M 碩士,4337,1904,2433,0,0,...,2,0,1,1,0,0,0,0,30 臺北市,1 一般
2,113,1,國立政治大學,N 職,M 碩士,1864,902,962,0,0,...,70,63,0,0,0,0,0,0,30 臺北市,1 一般
3,113,1,國立政治大學,D 日,B 學士,9801,3991,5810,0,0,...,0,0,0,0,0,0,375,468,30 臺北市,1 一般
4,113,2,國立清華大學,D 日,D 博士,1661,1158,503,0,0,...,127,62,93,49,125,55,0,0,18 新竹市,1 一般


## 3. 觀察 — 不同年度欄位數不一樣

**Schema (資料表結構)**：講白話就是「這份表格有哪些欄位」。

教育部多年來統計欄位有變動：105–106 學年度比較少欄,107 之後加了「縣市名稱」、113 又新增「華語先修生」欄位。

**問題**：跨年度合併時要怎麼處理這種「欄位不一致」？

In [ ]:
# 抓 105 與 113 學年度，比較欄位差異
df_105 = pd.read_csv(StringIO(requests.get(MOE_URL.format(year=105),
                                            timeout=30, headers=HEADERS)
                                .content.decode("utf-8-sig")))
print(f"105 學年度欄位數: {len(df_105.columns)}")
print(f"113 學年度欄位數: {len(df_113.columns)}")
print(f"\n113 有但 105 沒有的欄位：")
#for c in df_113.columns:
#    if c not in df_105.columns:
#        print(f"  - {c}")
df_105.columns       

105 學年度欄位數: 22
113 學年度欄位數: 28

113 有但 105 沒有的欄位：


TypeError: 'Index' object is not callable

## 4. 抓 9 個學年度 (105–113) 並合併

**`pd.concat`** 的兩個重要參數:
- `ignore_index=True`：重新編號 0..N-1
- `sort=False`：保留欄位原本順序，不要按字母排

缺欄位的年份會自動被填成 `NaN`,這就是 schema alignment (對齊欄位結構)。

In [15]:
parts = []
for y in range(105, 114):
    r = requests.get(MOE_URL.format(year=y), timeout=30, headers=HEADERS)
    r.raise_for_status()
    one_year = pd.read_csv(StringIO(r.content.decode("utf-8-sig")))
    one_year["學年度"] = y   # 確保都有「學年度」欄
    parts.append(one_year)
    print(f"  {y} 學年度: {one_year.shape}")

df_raw = pd.concat(parts, ignore_index=True, sort=False)
print(f"\n合併後: {df_raw.shape}")

  105 學年度: (851, 23)
  106 學年度: (838, 23)
  107 學年度: (812, 26)
  108 學年度: (794, 26)
  109 學年度: (820, 26)
  110 學年度: (791, 26)
  111 學年度: (783, 26)
  112 學年度: (764, 26)
  113 學年度: (737, 28)

合併後: (7190, 44)


## 5. 清理 — 「觀察 → 動作 → 代價」

本份資料有 4 件事要做：
1. 「縣市名稱」內容是 `"30 臺北市"` → 用 regex 拆出 `city_name`
2. 「總計」可能因千分號被讀成字串 → `to_numeric(errors='coerce')`
3. 105–106 學年度沒有「總計」欄 → 從各年級男/女加總補上
4. 公私立沒有專屬欄位 → 從學校名稱推導 ("國立"=公,「市立」=公,...)

In [ ]:
df = df_raw.copy()

# (1) 縣市名稱："30 臺北市" → 拆出 city_name
df["city_name"] = df["縣市名稱"].astype(str).str.extract(r"(?:\d+\s*)?(\S+)$")[0]

# (2) 把所有「人數」欄位轉 numeric (含千分號處理)
#     兩種 schema 並存：
#       105–106: 一年級男生 / 一年級女生 / ... / 延修生男生 / 延修生女生  (結尾「男生」「女生」)
#       107+   : 男生計 / 女生計 / 總計
#       113+   : 一年級男 / 一年級女 / ... / 華語先修生男 / 華語先修生女   (結尾「男」「女」)
count_cols = [c for c in df.columns
              if c.endswith(("男", "女", "男生", "女生", "男生計", "女生計", "總計"))]
for c in count_cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )

# (3) 「總計」NaN 補值 — 從各年級男/女欄位 row-wise 加總
#     105–106 學年度根本沒有「總計」欄(concat 後變 NaN),要用年級欄位補回來。
grade_cols = [c for c in df.columns if c.endswith(("男", "女", "男生", "女生"))]
row_total = df[grade_cols].sum(axis=1, skipna=True)

if "總計" in df.columns:
    df["總計"] = df["總計"].fillna(row_total)
else:
    df["總計"] = row_total

# (4) 公私立推導 (rule-based)
df["sector"] = df["學校名稱"].astype(str).apply(
    lambda s: "公立" if s.startswith(("國立", "市立", "省立", "國防", "警察")) else "私立"
)

print(f"清理完成: {df.shape}")
print(f"總計 NaN 數量: {df['總計'].isna().sum()}")
df[["學年度", "學校名稱", "city_name", "sector", "總計"]].head()

## 6. 描述性統計 — 健康檢查

In [9]:
print(f"列數          : {len(df)}")
print(f"學年度範圍     : {df['學年度'].min()} → {df['學年度'].max()}")
print(f"unique 學校   : {df['學校名稱'].nunique()}")
print(f"unique 縣市   : {df['city_name'].nunique()}")
print(f"\n每年總學生數：")
agg = df.groupby("學年度")["總計"].sum()
print(agg.to_string())
print(f"\n9 年累計減少：{agg.iloc[0] - agg.iloc[-1]:,.0f} 人"
      f" ({(agg.iloc[0]-agg.iloc[-1])/agg.iloc[0]*100:.1f}%)")

列數          : 7190
學年度範圍     : 105 → 113
unique 學校   : 169
unique 縣市   : 21

每年總學生數：
學年度
105          0.0
106          0.0
107    1244822.0
108    1213172.0
109    1203460.0
110    1185830.0
111    1140089.0
112    1094829.0
113    1074365.0

9 年累計減少：-1,074,365 人 (-inf%)


C:\Users\audachang\AppData\Local\Temp\ipykernel_15208\2188061269.py:9: RuntimeWarning: divide by zero encountered in scalar divide
  f" ({(agg.iloc[0]-agg.iloc[-1])/agg.iloc[0]*100:.1f}%)")


## 7. 圖 1 — 總學生數逐年變化 (line + annotation)

回答：「整體是不是真的在下降？」

加 annotation 標出 105 學年度的起點 — 政策報告中加 annotation 比單純看曲線更有說服力。

In [10]:
agg = df.groupby("學年度")["總計"].sum().reset_index()

fig = px.line(
    agg, x="學年度", y="總計", markers=True,
    title="台灣大專校院總學生數 — 105–113 學年度",
    labels={"學年度": "Academic year", "總計": "Total students"},
    color_discrete_sequence=["#3b82f6"],
)
fig.add_annotation(
    x=agg["學年度"].iloc[0], y=agg["總計"].iloc[0],
    text="少子化骨牌效應起點", showarrow=True, arrowhead=2,
    bgcolor="rgba(255,255,255,0.9)", bordercolor="orange",
)
fig.show()

## 8. 圖 2 — 公立 vs. 私立 (stacked bar)

回答：「下降集中在哪個 sector？」

用 stacked bar 把組成攤開,可以一眼看出私立 (橘) 縮得比公立 (藍) 快。

In [11]:
agg = df.groupby(["學年度", "sector"])["總計"].sum().reset_index()

fig = px.bar(
    agg, x="學年度", y="總計", color="sector", barmode="stack",
    title="公立 vs. 私立 大專學生數變化",
    labels={"學年度": "Academic year", "總計": "Total students",
            "sector": "Sector"},
    color_discrete_map={"公立": "#1d4ed8", "私立": "#f97316"},
)
fig.show()

# 數字驗證
print("\n各 sector 9 年變化:")
wide = agg.pivot(index="學年度", columns="sector", values="總計")
for col in wide.columns:
    change = wide[col].iloc[-1] - wide[col].iloc[0]
    pct = change / wide[col].iloc[0] * 100
    print(f"  {col}: {change:+,.0f} ({pct:+.1f}%)")


各 sector 9 年變化:
  公立: +451,397 (+inf%)
  私立: +622,968 (+inf%)


C:\Users\audachang\AppData\Local\Temp\ipykernel_15208\1011416105.py:17: RuntimeWarning:

divide by zero encountered in scalar divide



## 9. 圖 3 — 113 學年度各縣市學生數 (horizontal bar)

回答：「最新一年,哪些縣市的大專生最多？」

In [12]:
latest = df[df["學年度"] == df["學年度"].max()]
agg = (latest.groupby("city_name")["總計"].sum()
              .sort_values(ascending=False).reset_index().head(15))

fig = px.bar(
    agg, x="總計", y="city_name", orientation="h",
    color="總計", color_continuous_scale="Tealgrn",
    title=f"{df['學年度'].max()} 學年度各縣市大專學生數 (Top 15)",
    labels={"總計": "Total students", "city_name": "City"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  coloraxis_showscale=False)
fig.show()

## 10. 儲存清理後資料

In [13]:
df.to_csv("moe_higher_ed.csv", index=False, encoding="utf-8-sig")
print("已存到 moe_higher_ed.csv")

已存到 moe_higher_ed.csv
